# Given a target epsilon,delta, and PE iteration T, return the noise_multiplier in PE histogram.

In [1]:
import scipy.optimize
from scipy.optimize import root_scalar
import numpy as np
import math

def delta_Gaussian(eps, mu):
    """Compute delta of Gaussian mechanism with shift mu or equivalently noise scale 1/mu.

    :param eps: The epsilon value
    :type eps: float
    :param mu: The mu value
    :type mu: float
    :return: The delta value
    :rtype: float
    """
    if mu == 0:
        return 0
    if np.isinf(np.exp(eps)):
        return 0
    return scipy.stats.norm.cdf(-eps / mu + mu / 2) - np.exp(eps) * scipy.stats.norm.cdf(-eps / mu - mu / 2)

def eps_Gaussian(delta, mu, max_epsilon):
    """Compute eps of Gaussian mechanism with shift mu or equivalently noise scale 1/mu.

    :param delta: The delta value
    :type delta: float
    :param mu: The mu value
    :type mu: float
    :param max_epsilon: The maximum epsilon value to search for
    :type max_epsilon: float
    """

    def f(x):
        return delta_Gaussian(x, mu) - delta

    return root_scalar(f, bracket=[0, max_epsilon], method="brentq").root

def compute_epsilon(noise_multiplier, num_steps, delta, max_epsilon=1e7):
    """Compute epsilon of Gaussian mechanism.

    :param noise_multiplier: The noise multiplier
    :type noise_multiplier: float
    :param num_steps: The number of steps
    :type num_steps: int
    :param delta: The delta value
    :type delta: float
    :param max_epsilon: The maximum epsilon value to search for, defaults to 1e7
    :type max_epsilon: float, optional
    :return: The epsilon value.
    :rtype: float
    """
    if noise_multiplier == 0:
        execution_logger.warning("Since noise_multiplier is 0, epsilon is INF.")
        return np.inf
    return eps_Gaussian(delta=delta, mu=np.sqrt(num_steps) / noise_multiplier, max_epsilon=max_epsilon)

def get_noise_multiplier(
    epsilon,
    num_steps,
    delta,
    min_noise_multiplier=1e-1,
    max_noise_multiplier=500,
    max_epsilon=1e7,
):
    """Get noise multiplier of Gaussian mechanism.

    :param epsilon: The epsilon value
    :type epsilon: float
    :param num_steps: The number of steps
    :type num_steps: int
    :param delta: The delta value
    :type delta: float
    :param min_noise_multiplier: The minimum noise multiplier to search for, defaults to 1e-1
    :type min_noise_multiplier: float, optional
    :param max_noise_multiplier: The maximum noise multiplier to search for, defaults to 500
    :type max_noise_multiplier: float, optional
    :param max_epsilon: The maximum epsilon value to search for, defaults to 1e7
    :type max_epsilon: float, optional
    """

    if epsilon == np.inf:
        return 0.0

    def objective(x):
        return (
            compute_epsilon(
                noise_multiplier=x,
                num_steps=num_steps,
                delta=delta,
                max_epsilon=max_epsilon,
            )
            - epsilon
        )

    output = root_scalar(objective, bracket=[min_noise_multiplier, max_noise_multiplier], method="brentq")

    if not output.converged:
        raise ValueError("Failed to converge")

    return output.root


def zcdp_delta(rho: float, eps: float) -> float:
    """Return the minimum value of delta such that rho-zCDP implies (epsilon, delta)-DP."""
    assert rho >= 0
    assert eps >= 0
    if rho == 0:
        return 0

    amin = 1.01
    amax = (eps + 1) / (2 * rho) + 2
    alpha = math.nan
    while amax - amin > 1e-10:
        alpha = (amin + amax) / 2
        derivative = (2 * alpha - 1) * rho - eps + math.log1p(-1.0 / alpha)
        if derivative < 0:
            amin = alpha
        else:
            amax = alpha
    delta = math.exp(
      (alpha - 1) * (alpha * rho - eps) + alpha * math.log1p(-1 / alpha)
    ) / (alpha - 1.0)
    return min(delta, 1.0)

def zcdp_gaussian_sigma(rho: float) -> float:
    """Minimum sigma such that the Gaussian mechanism satisfies rho-zCDP."""
    # rho = 0.5 / sigma^2
    return math.sqrt(0.5 / rho)

def zcdp_rho(eps: float, delta: float) -> float:
    """Return the maximum value of rho such that rho-zCDP implies (epsilon, delta)-DP."""
    assert eps >= 0
    assert delta > 0
    if delta >= 1:
        return 0.0
    rhomin = 0.0
    rhomax = eps + 1
    while rhomax - rhomin > 1e-10:
        rho = (rhomin + rhomax) / 2
        if zcdp_delta(rho, eps) <= delta:
            rhomin = rho
        else:
            rhomax = rho
    return rhomin


def zcdp_eps(rho: float, delta: float) -> float:
    """Return the minimum value of epsilon such that rho-zCDP implies (epsilon, delta)-DP."""
    assert rho >= 0
    assert delta > 0
    if delta >= 1 or rho == 0:
        return 0.0
    epsmin = 0.0
    epsmax = rho + 2 * math.sqrt(rho * math.log(1 / delta))
    while epsmax - epsmin > 1e-10:
        eps = (epsmin + epsmax) / 2
        if zcdp_delta(rho, eps) <= delta:
            epsmax = eps
        else:
            epsmin = eps
    return epsmax


def noise_calibration_PE(epsilon,delta,steps):
    noise_multiplier = get_noise_multiplier(
        epsilon = epsilon,
        num_steps = steps,
        delta = delta,
        min_noise_multiplier=1e-3,
        max_noise_multiplier=500,
        max_epsilon=1e7,
    )
    print(f'For eps={epsilon}, PE noise multiplier is: {noise_multiplier}')
    return noise_multiplier

def noise_calibaration_MAPLE(epsilon,delta,ratio,PE_steps):
    rho = zcdp_rho(eps = epsilon, delta = delta)
    print(f'For eps={epsilon}, Total rho is: {rho}')
    
    # split rho according to ratio, where ratio is the fraction of rho that is for AIM
    rho_aim = rho * ratio
    rho_pe = rho * (1 - ratio)
    
    # calibrate noise for PE part
    noise_multiplier = zcdp_gaussian_sigma(rho_pe/PE_steps)
    
    print(f'The rho_aim is: {rho_aim}, the noise_multiplier for PE part with {PE_steps} iteration is: {noise_multiplier}')
    return rho_aim, noise_multiplier

# Calibrating noises

In [4]:
import pandas as pd

# # This is for biorxiv
# df_trn = pd.read_csv('./biorxiv/biorxiv_train_metadata.csv')
# # This is for openreview
df_trn = pd.read_csv('./openreview/train_dataset_with_metadata_v4.csv')
# # This is for PMC
# df_trn = pd.read_csv('./pmc/PMC_V2_train_textonly.csv')

delta = 1/len(df_trn)

for eps in [4.0,2.0,1.0]:
    # Remember to change the number of steps if you're not using 10 PE iterations.
    noise_multiplier = noise_calibration_PE(epsilon=eps,delta=delta,steps=10)
    

For eps=4.0, PE noise multiplier is: 3.000778920378073
For eps=2.0, PE noise multiplier is: 5.4187020358260325
For eps=1.0, PE noise multiplier is: 9.935410662820491


/var/tmp/ipykernel_1561978/711548664.py:18: RuntimeWarning: overflow encountered in exp
  if np.isinf(np.exp(eps)):


# AugPE Noise Multiplier table (steps=10)

## Biorxiv
For eps=4.0, PE noise multiplier is: 3.214362874574832

For eps=2.0, PE noise multiplier is: 5.872135064401613

For eps=1.0, PE noise multiplier is: 10.889585799108582

## Openreview
For eps=4.0, PE noise multiplier is: 3.000778920378073

For eps=2.0, PE noise multiplier is: 5.418702035826035

For eps=1.0, PE noise multiplier is: 9.935410662820507


## PMC
For eps=4.0, PE noise multiplier is: 3.5575676151673346

For eps=2.0, PE noise multiplier is: 6.597507733016656

For eps=1.0, PE noise multiplier is: 12.408984498128014

## Noise calibration using zCDP. For MAPLE we can only use this.

In [6]:
import pandas as pd

# # his is for biorxiv
# df_trn = pd.read_csv('./biorxiv/biorxiv_train_metadata.csv')
# # This is for openreview
df_trn = pd.read_csv('./openreview/train_dataset_with_metadata_v4.csv')
# # This is for PMC
# df_trn = pd.read_csv('./pmc/PMC_V2_train_textonly.csv')

delta = 1/len(df_trn)
for eps in [4.0,2.0,1.0]:
    noise_calibaration_MAPLE(epsilon=eps,delta=delta,ratio=0.1,PE_steps=3)

For eps=4.0, Total rho is: 0.47335957184259314
The rho_aim is: 0.047335957184259314, the noise_multiplier for PE part with 3 iteration is: 1.876414555237503
For eps=2.0, Total rho is: 0.1424762885144446
The rho_aim is: 0.014247628851444462, the noise_multiplier for PE part with 3 iteration is: 3.4202124076206792
For eps=1.0, Total rho is: 0.04165570123586804
The rho_aim is: 0.0041655701235868035, the noise_multiplier for PE part with 3 iteration is: 6.325387702318415


# AugPE Noise Multiplier table

## biorxiv
For eps=4.0, Total rho is: 0.4177113722835202

AIM rho: 0.04177113722835202
For 5 iterations, PE noise per iter: 2.5787580697142025

===

For eps=2.0, Total rho is: 0.12320561576052569

AIM rho: 0.01232056157605257
For 5 iterations, PE noise per iter: 4.7482491352519505

==

For eps=1.0, Total rho is: 0.0353213325724937

AIM rho: 0.00353213325724937
For 3 iterations, PE noise per iter: 6.869194837509232

## Openreview

For eps=4.0, Total rho is: 0.47335957184259314

AIM rho: 0.047335957184259314
For 3 iterations, PE noise per iter: 1.876414555237503

===

For eps=2.0, Total rho is: 0.1424762885144446

AIM rho: 0.01424762885144446
For 3 iterations, PE noise per iter: 3.4202124076206792

===

For eps=1.0, Total rho is: 0.04165570123586804

AIM rho: 0.0041655701235868035
For 1 iterations, PE noise per iter: 3.6519642926622855

## PMC

For eps=4.0, Total rho is: 0.3468809820333263

AIM rho: 0.03468809820333263
For 3 iterations, PE noise per iter: 2.1919675966960335

===

For eps=2.0, Total rho is: 0.09964905318338424

AIM rho: 0.009964905318338424
For 3 iterations, PE noise per iter: 4.08966548735772

===

For eps=1.0, Total rho is: 0.027872676379047334

AIM rho: 0.0027872676379047334
For 3 iterations, PE noise per iter: 7.732769033384689